# 1. Set Paths

In [ ]:
import os
from pathlib import Path

In [ ]:
# DATASET_PATH = Path("/data/pt_03068/data/in_vivo")
# DATASET_PATH = Path("/data/pt_02262/data/TH_bids")
# DATASET_PATH = Path("/data/pt_02262/data/liege_data")
DATASET_PATH = Path("/data/pt_03187/data/in_vivo")

SOURCE_PATH = DATASET_PATH / "source"
PREPARED_PATH = DATASET_PATH / "temp"
BIDSIFIED_PATH = DATASET_PATH / "bids"
# RESOURCES_PATH = BIDSIFIED_PATH / "code" / "resources"
WORKING_DIR = Path.cwd()

In [ ]:
print("Dataset path:", DATASET_PATH.is_dir())
print("Source path:", SOURCE_PATH.is_dir())
print("Prepared path:", PREPARED_PATH.is_dir())
print("Bidsified path:", BIDSIFIED_PATH.is_dir())
# print("Resources path:", RESOURCES_PATH.is_dir())
print("Working directory:", WORKING_DIR)

# 2. Initialize `bidsme` and get the `logger` object
which which will control the logging of all bidsme functions:will control the logging of all bidsme functions:

In [ ]:
import bidsme
logger = bidsme.init()

# 3. Prepare data set for bidsification

In [ ]:
#help(bidsme.prepare)

In [ ]:
logger.setLevel("INFO")
bidsme.prepare(str(SOURCE_PATH), str(PREPARED_PATH), 
               data_dirs={
                        ### fine-grained directory specification example
                        # "nii/localizer*":"MRI",
                        # "nii/calc_shims_40mm*":"MRI",
                        # "nii/ernst_kp_mtflash3d_*_0p5_sag_*":"MRI",
                        
                        ### Traveling heads data / HISTOPARK data
                        # "nii":"MRI", # files directly in the nii folder
                        # "nii/*":"MRI", # all files in subfolders of nii
                        # "nii_dcm2niix_dwi/*":"MRI",
                        
                        # Traveling heads data after XA update
                        # "nii_dcm2niix/*":"MRI",

                        ### Liege IronSleep data
                        # "nii/*":"MRI"

                        ### XALD data
                        # "nii_dcm2niix/*":"MRI",

                        ### HISTOPARK data (Prisma (SPM Dicom Import) and Terra.X (dcm2niix) data simultaneously)
                        "nii/*":"MRI", # all files in subfolders of nii
                        "nii_dcm2niix/*":"MRI",
                          }, 
               # plugin_file = str(WORKING_DIR / "plugins_bidsme" / "plugin_prepare_nk.py"),
               # plugin_file = str(WORKING_DIR / "plugins_bidsme" / "TerraX_data" / "plugin_prepare_terrax_dcm2niix_nk.py"),
               plugin_file = str(WORKING_DIR / "plugins_bidsme" / "combined_dcm2niix_dcmImport" /  "plugin_prepare_combined_dcm2niix_DcmImport_nk.py"),
               part_template = str(WORKING_DIR / "supplementary" / "table_templates" / "participants_nk.json"),
               plugin_opt = {"sessions_tsv_template": str(WORKING_DIR / "supplementary" / "table_templates" / "sessions_nk.json")}, 
               ses_skip_dir=True,
               # sub_list=["sub-021"], # only run on specified subjects (must be specified in BIDS notation)
               )
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

> **Note**: **apparently, it is not possible to specify a specific subset of sessions**
> + If the user wishes to rename subjects and/or sessions, it can be done with plug-in functions ```SubjectEP``` and ```SessionEP``` or by renaming directly folders in the prepared dataset.

# 4. Create the bidsmap.yaml

+ most tedious part of the process

In [ ]:
# help(bidsme.mapper)

In [ ]:
# PLUGIN_BIDS = WORKING_DIR / "plugins_bidsme" / "plugin_bidsify_nk.py"
# PLUGIN_BIDS = WORKING_DIR / "plugins_bidsme" / "TerraX_data" / "plugin_bidsify_terrax_dcm2niix_nk.py"

PLUGIN_BIDS = str(WORKING_DIR / "plugins_bidsme" / "combined_dcm2niix_dcmImport" / "plugin_bidsify_combined_dcm2niix_DcmImport_nk.py")
MAP_FILE = str(BIDSIFIED_PATH / "code" / "bidsme" / "bidsmap.yaml")

In [ ]:
bidsme.mapper(str(PREPARED_PATH), str(BIDSIFIED_PATH), plugin_file=str(PLUGIN_BIDS), 
              bidsmapfile=str(MAP_FILE),
              plugin_opt={"bidsmap_step": True},
              sub_list=["sub-021"],
              )
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

+ open the created yaml file in VS Code
+ fix each warning/error, save the file, and repeat the code block above
    - find help at in the [Jupyter Notebooks in the bidsme tutorial](https://github.com/CyclotronResearchCentre/bidsme_tutorial) or in the docs [under Bidsmap creation](https://github.com/CyclotronResearchCentre/bidsme/blob/dev/doc/creating_map.md)
+ resume until there are no warnings left

> **Note:** to find a specific string in a file, use the command ```cat file.json | grep -i "string"``` **or** use ```less file.json``` and search using ```/string``` (```n``` will take you to the next entry & ```shift+n``` to the previous one; ```-I``` for case-insensitive)

> **Note:** Bidsme allow some limited transformation of data retrieved from header, these transformations are called actions and are defined in function ```action_value``` in file ```INSTALLATION_PATH/bidsme/Modules/common.py```.

```
Accepted actions:
    "": no action, return value
    int: cast value to int
    float: cast value to float
    str: cast value to string
    format<parameters>: apply python3 formatting
        mini-language to value, {:<parameters>}.format(value)
    scale<int>: apply a 10-based scale to value,
        value ** <int>
    mult<float>: multiply value
    div<float>: divide value
    round<int>: round value to given precision
```


+ The naming schema and sidecar json fields for a given modality (in this case MRI) are defined in $INSTALLATION_PATH/bidsme/Modules/MRI/_MRI.py. The list of entities is stored in modalities dictionary. If an image belongs for example to anat, bidsme will load the list of entities from modalities["anat"].

+ The optional model field will foce to use different list of entities from modalities dictionary. We will use the models extensively, while creating map for MPM part of the examle dataset.


# 5. Bidsification of the data set

In [ ]:
MAP_FILE = str(BIDSIFIED_PATH / "code" / "bidsme" / "bidsmap.yaml")
# PLUGIN_FILE_BIDS = str(WORKING_DIR / "plugins_bidsme" / "plugin_bidsify_nk.py")
# PLUGIN_FILE_BIDS = str(WORKING_DIR / "plugins_bidsme" / "TerraX_data" / "plugin_bidsify_terrax_dcm2niix_nk.py")
PLUGIN_FILE_BIDS = str(WORKING_DIR / "plugins_bidsme" / "combined_dcm2niix_dcmImport" / "plugin_bidsify_combined_dcm2niix_DcmImport_nk.py")
if not os.path.exists(PLUGIN_FILE_BIDS):
    raise FileNotFoundError(f"Plugin file {PLUGIN_FILE_BIDS} not found.")

-b = mapping file, --plugin = plugin

In [ ]:
# !bidsme bidsify -help

In [ ]:
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS --participants 'sub-011'
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS --skip-existing

In [ ]:
# help(bidsme.bidsify)

In [ ]:
bidsme.bidsify(str(PREPARED_PATH), str(BIDSIFIED_PATH), 
               bidsmapfile=str(MAP_FILE), 
               plugin_file=str(PLUGIN_FILE_BIDS),
               plugin_opt={"bidsmap_step": False},
                # sub_list=["sub-001"]
)
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

## Hints


The `004-al_mtflash3d_PDw` and `005-al_mtflash3d_PDw` are
the anatomical images
(suffix -- `MPM`)
taken using several echo times (`echo-1` ... `echo-6`),
and splitted into magnitude and phase components (`part-mag` and `part-phase`).
Additionaly, as for the PD-weighted images, the MT pulse wasn't used, we will
add the `mt-off` entity.
We will also add `flip-1` to the name, to mark that PDw images uses different
flip angle from T
So the final name will become:
`anat/sub-001_ses-s01530_acq-PDw_echo-1_mt-off_part-mag_MPM.nii`.
`anat/sub-001_ses-s01530_acq-PDw_echo-1_flip-1_mt-off_part-mag_MPM.nii`

The `002-al_mtflash3d_sensArray` and `003-al_mtflash3d_sensBody`
are [B1 fieldmaps](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#rb1cor-specific-notes)
(suffix -- `RB1COR`),
taken for the PD-weighted images, using head and body coils
(`acq-headPDw` and `acq-bodyPDw`).
So their names will be simply: `fmap/sub-001_ses-s01530_acq-headPDw_RB1COR.nii`.

The similar names can be applied to T1w and MTw images and corresponding fieldmaps,
using corresponding `acq-` entities `acq-T1w` and `acq-MTw`.
For MTw images we alse need to use the `mt-on` entity.

Finally, `014-al_B1mapping` is the
[global B1 map](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#tb1epi-specific-notes)
(suffix -- `TB1EPI`)
sets of images, taken with two echo times (`echo-1`, `echo-2`)
and several flip angles (`flip-01`, ... `flip-08`).
So the final name will become: `fmap/sub-001_ses-s01530_echo-1_flip-01_TB1EPI`